## Imports and Function Definitions

In [1]:
import os
import sys
import json
import time
import gc
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1337

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU found — running on CPU")

print(f"\nDevice  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")
print(f"Seed    : {SEED}")

GPU : NVIDIA L4
VRAM: 23.6 GB

Device  : cuda
PyTorch : 2.6.0+cu124
Seed    : 1337


In [2]:
# =============================================================================
# NIH CHESTXRAY14 CLINICAL UTILITY - CONFIGURATION
# =============================================================================
# Multi-label binary classification: 11 pathology classes
# Fixed patient-level train / val / test splits (no CV)
#
# Conditions tested:
#   raw, gs50, gs40, gs30, gs20, gs10, gs0  (reverse order — max drop first)
#
# Evaluation modes:
#   Adaptive : train and eval on same condition
#   Blind    : train on GS, eval on raw
#
# Task: multi-label BCE (sigmoid per class, independent binary heads)
#   - per-class binary AUC, macro-averaged as primary metric
#   - per-class AUC logged at test time
#   - gender-stratified AUC at test time
# =============================================================================

warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*deprecated.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.GradScaler.*deprecated.*")

# =============================================================================
# PATHS
# =============================================================================
PROJECT_ROOT = Path('.')
DATA_DIR     = PROJECT_ROOT / 'data' / 'chest' / 'npz'
META_DIR     = PROJECT_ROOT / 'data' / 'metadata'
RESULTS_DIR  = PROJECT_ROOT / 'results' / 'nih_cxr' / 'utility'
MODELS_DIR   = PROJECT_ROOT / 'models'  / 'nih_cxr' / 'utility'
CACHE_DIR    = PROJECT_ROOT / 'cache'   / 'nih_cxr' / 'utility'
TENSOR_DIR   = CACHE_DIR / 'tensors'

for d in [RESULTS_DIR, MODELS_DIR, CACHE_DIR, TENSOR_DIR]:
    d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Output files
PER_RUN_CSV = RESULTS_DIR / 'per_run_results.csv'
SUMMARY_CSV = RESULTS_DIR / 'summary_results.csv'

# =============================================================================
# DEVICE & REPRODUCIBILITY
# =============================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 1337
seed_everything(SEED)

# =============================================================================
# EXPERIMENT GRID
# =============================================================================
ARCHITECTURES = ['resnet18', 'densenet121']
CONDITIONS    = ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']  # reverse

GS_BATCH_SIZE = 100
GS_ITERATIONS = 50

# =============================================================================
# TRAINING HYPERPARAMETERS
# =============================================================================
BATCH_SIZE         = 128
NUM_EPOCHS         = 100
PATIENCE           = 10
LR_PRETRAINED      = 5e-5
LR_SCRATCH         = 8e-5
WD_PRETRAINED      = 1e-4
WD_SCRATCH         = 5e-4
DROPOUT_P          = 0.5
HEAD_LR_MULTIPLIER = 5.0

# =============================================================================
# CLASS SETUP
# =============================================================================
with open(META_DIR / 'class_to_idx.json') as f:
    CLASS_TO_IDX = json.load(f)

IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES  = len(CLASS_TO_IDX)  # 11

SHORT_NAMES = {
    'No Finding':         'NoFind',
    'Infiltration':       'Infilt',
    'Atelectasis':        'Atelec',
    'Effusion':           'Effus',
    'Nodule':             'Nodule',
    'Pneumothorax':       'PneuTx',
    'Mass':               'Mass',
    'Consolidation':      'Consol',
    'Pleural_Thickening': 'PlThck',
    'Cardiomegaly':       'CardMeg',
    'Emphysema':          'Emphy',
}
SHORT_LIST = [SHORT_NAMES[IDX_TO_CLASS[i]] for i in range(NUM_CLASSES)]

# Split conditions for tensor builder
SPLIT_CONDITIONS = {
    'train': CONDITIONS,
    'val'  : CONDITIONS,
    'test' : ['raw'],
}

print("=" * 70)
print("NIH CHESTXRAY14 UTILITY - CONFIGURATION")
print("=" * 70)
print(f"Device             : {DEVICE}")
print(f"Seed               : {SEED}")
print(f"Task               : multi-label BCE (11 independent binary heads)")
print(f"Data dir           : {DATA_DIR}")
print(f"Results dir        : {RESULTS_DIR}")
print(f"Batch size         : {BATCH_SIZE}")
print(f"Max epochs         : {NUM_EPOCHS}")
print(f"Patience           : {PATIENCE}")
print(f"LR pretrained      : {LR_PRETRAINED} (head) / "
      f"{LR_PRETRAINED/HEAD_LR_MULTIPLIER} (body)")
print(f"LR scratch         : {LR_SCRATCH}")
print(f"WD pretrained      : {WD_PRETRAINED}")
print(f"WD scratch         : {WD_SCRATCH}")
print(f"Dropout            : {DROPOUT_P}")
print(f"Head LR multiplier : {HEAD_LR_MULTIPLIER}")
print(f"Num classes        : {NUM_CLASSES}")
print(f"Architectures      : {ARCHITECTURES}")
print(f"Conditions         : {CONDITIONS}")
print("=" * 70)
print(f"\nClass mapping:")
for cls, idx in CLASS_TO_IDX.items():
    print(f"  {idx:2d}  {SHORT_NAMES[cls]:<10}  ({cls})")

NIH CHESTXRAY14 UTILITY - CONFIGURATION
Device             : cuda
Seed               : 1337
Task               : multi-label BCE (11 independent binary heads)
Data dir           : data/chest/npz
Results dir        : results/nih_cxr/utility
Batch size         : 128
Max epochs         : 100
Patience           : 10
LR pretrained      : 5e-05 (head) / 1e-05 (body)
LR scratch         : 8e-05
WD pretrained      : 0.0001
WD scratch         : 0.0005
Dropout            : 0.5
Head LR multiplier : 5.0
Num classes        : 11
Architectures      : ['resnet18', 'densenet121']
Conditions         : ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']

Class mapping:
   0  NoFind      (No Finding)
   1  Infilt      (Infiltration)
   2  Atelec      (Atelectasis)
   3  Effus       (Effusion)
   4  Nodule      (Nodule)
   5  PneuTx      (Pneumothorax)
   6  Mass        (Mass)
   7  Consol      (Consolidation)
   8  PlThck      (Pleural_Thickening)
   9  CardMeg     (Cardiomegaly)
  10  Emphy       (Emph

In [3]:
# =============================================================================
# LOAD NPZ SPLITS
# =============================================================================

def labels_to_multihot(labels: np.ndarray, num_classes: int) -> np.ndarray:
    """
    Convert integer class labels to multi-hot binary vectors.
    Since our dataset is single-label filtered, each row has exactly one 1.

    Args:
        labels      : (N,) int64 class indices
        num_classes : number of classes

    Returns:
        multihot    : (N, num_classes) float32
    """
    multihot = np.zeros((len(labels), num_classes), dtype=np.float32)
    multihot[np.arange(len(labels)), labels] = 1.0
    return multihot


def load_split(
    split: str
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a split from NPZ.

    Returns:
        images      : (N, 224, 224)       float32  normalized [0, 1]
        labels_int  : (N,)                int64    original class indices
        labels_mh   : (N, NUM_CLASSES)    float32  multi-hot binary vectors
        subject_ids : (N,)                int64
        genders     : (N,)                int8     M=1, F=0
    """
    path = DATA_DIR / f'{split}.npz'
    data = np.load(path)

    images      = data['images'].astype(np.float32)
    labels_int  = data['labels'].astype(np.int64)
    subject_ids = data['subject_ids'].astype(np.int64)
    genders     = data['genders'].astype(np.int8)
    data.close()

    labels_mh = labels_to_multihot(labels_int, NUM_CLASSES)

    return images, labels_int, labels_mh, subject_ids, genders


print("[LOAD] Loading splits...")
TRAIN_IMAGES, TRAIN_LABELS, TRAIN_LABELS_MH, TRAIN_SIDS, TRAIN_GENDERS = load_split('train')
VAL_IMAGES,   VAL_LABELS,   VAL_LABELS_MH,   VAL_SIDS,   VAL_GENDERS   = load_split('val')
TEST_IMAGES,  TEST_LABELS,  TEST_LABELS_MH,  TEST_SIDS,  TEST_GENDERS   = load_split('test')

# =============================================================================
# SANITY CHECKS
# =============================================================================
print("\n=== SPLIT SUMMARY ===")
for name, imgs, labs_int, labs_mh, sids, gens in [
    ('Train', TRAIN_IMAGES, TRAIN_LABELS, TRAIN_LABELS_MH, TRAIN_SIDS, TRAIN_GENDERS),
    ('Val',   VAL_IMAGES,   VAL_LABELS,   VAL_LABELS_MH,   VAL_SIDS,   VAL_GENDERS),
    ('Test',  TEST_IMAGES,  TEST_LABELS,  TEST_LABELS_MH,  TEST_SIDS,  TEST_GENDERS),
]:
    n_subj = len(np.unique(sids))
    m_pct  = (gens == 1).mean() * 100
    f_pct  = (gens == 0).mean() * 100
    print(f"\n  {name}:")
    print(f"    Images shape : {imgs.shape}  dtype={imgs.dtype}")
    print(f"    Labels (int) : {labs_int.shape}  dtype={labs_int.dtype}")
    print(f"    Labels (mh)  : {labs_mh.shape}   dtype={labs_mh.dtype}  "
          f"sum/row={labs_mh.sum(axis=1).mean():.1f}  "
          f"(should be 1.0 — single-label)")
    print(f"    Patients     : {n_subj:,}")
    print(f"    Gender       : M={m_pct:.1f}%  F={f_pct:.1f}%")
    print(f"    Pixel range  : [{imgs.min():.3f}, {imgs.max():.3f}]  "
          f"mean={imgs.mean():.3f}  std={imgs.std():.3f}")
    print(f"    Class dist   :")
    unique, counts = np.unique(labs_int, return_counts=True)
    for idx, cnt in zip(unique, counts):
        print(f"      {idx:2d} {SHORT_NAMES[IDX_TO_CLASS[idx]]:<10} {cnt:>5,}  "
              f"({cnt/len(labs_int)*100:.1f}%)")

# Patient leakage check
train_set = set(TRAIN_SIDS)
val_set   = set(VAL_SIDS)
test_set  = set(TEST_SIDS)
assert len(train_set & val_set)  == 0, "LEAKAGE: train/val"
assert len(train_set & test_set) == 0, "LEAKAGE: train/test"
assert len(val_set   & test_set) == 0, "LEAKAGE: val/test"
print("\n✓ No patient overlap across splits")

# Memory footprint
total_mb = (
    TRAIN_IMAGES.nbytes + VAL_IMAGES.nbytes + TEST_IMAGES.nbytes +
    TRAIN_LABELS_MH.nbytes + VAL_LABELS_MH.nbytes + TEST_LABELS_MH.nbytes
) / 1e6
print(f"✓ Total in-memory footprint: {total_mb:.0f} MB")

[LOAD] Loading splits...

=== SPLIT SUMMARY ===

  Train:
    Images shape : (24654, 224, 224)  dtype=float32
    Labels (int) : (24654,)  dtype=int64
    Labels (mh)  : (24654, 11)   dtype=float32  sum/row=1.0  (should be 1.0 — single-label)
    Patients     : 9,436
    Gender       : M=55.8%  F=44.2%
    Pixel range  : [0.000, 1.000]  mean=0.495  std=0.247
    Class dist   :
       0 NoFind     4,276  (17.3%)
       1 Infilt     6,661  (27.0%)
       2 Atelec     2,955  (12.0%)
       3 Effus      2,774  (11.3%)
       4 Nodule     1,916  (7.8%)
       5 PneuTx     1,481  (6.0%)
       6 Mass       1,507  (6.1%)
       7 Consol       944  (3.8%)
       8 PlThck       776  (3.1%)
       9 CardMeg      739  (3.0%)
      10 Emphy        625  (2.5%)

  Val:
    Images shape : (3725, 224, 224)  dtype=float32
    Labels (int) : (3725,)  dtype=int64
    Labels (mh)  : (3725, 11)   dtype=float32  sum/row=1.0  (should be 1.0 — single-label)
    Patients     : 1,346
    Gender       : M=58.6% 

In [4]:
# =============================================================================
# MODEL FACTORY
# =============================================================================

def get_weights_enum(arch: str):
    if arch == 'resnet18':
        return models.ResNet18_Weights.DEFAULT
    if arch == 'densenet121':
        return models.DenseNet121_Weights.DEFAULT
    raise ValueError(f"Unknown architecture: {arch}")


def get_model(arch: str = 'resnet18', weights=None) -> nn.Module:
    is_pretrained = weights is not None

    if arch == 'resnet18':
        model     = models.resnet18(weights=weights)
        old_conv  = model.conv1
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2,
                                padding=3, bias=False)
        if is_pretrained:
            with torch.no_grad():
                model.conv1.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )
        else:
            nn.init.kaiming_normal_(model.conv1.weight,
                                    mode='fan_out', nonlinearity='relu')

        if is_pretrained:
            for name, param in model.named_parameters():
                if not any(name.startswith(s)
                           for s in ['layer4', 'fc']):
                    param.requires_grad = False

        num_ftrs    = model.fc.in_features
        head_linear = nn.Linear(num_ftrs, NUM_CLASSES)
        nn.init.xavier_uniform_(head_linear.weight)
        nn.init.zeros_(head_linear.bias)
        model.fc = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            head_linear
        )

    elif arch == 'densenet121':
        model    = models.densenet121(weights=weights,
                                      memory_efficient=True)
        old_conv = model.features.conv0
        model.features.conv0 = nn.Conv2d(1, 64, kernel_size=7,
                                          stride=2, padding=3,
                                          bias=False)
        if is_pretrained:
            with torch.no_grad():
                model.features.conv0.weight.copy_(
                    old_conv.weight.mean(dim=1, keepdim=True)
                )
        else:
            nn.init.kaiming_normal_(model.features.conv0.weight,
                                    mode='fan_out', nonlinearity='relu')

        if is_pretrained:
            for name, param in model.named_parameters():
                if not any(name.startswith(s) for s in [
                    'features.denseblock3',
                    'features.denseblock4',
                    'classifier'
                ]):
                    param.requires_grad = False

        num_ftrs    = model.classifier.in_features
        head_linear = nn.Linear(num_ftrs, NUM_CLASSES)
        nn.init.xavier_uniform_(head_linear.weight)
        nn.init.zeros_(head_linear.bias)
        model.classifier = nn.Sequential(
            nn.Dropout(p=DROPOUT_P),
            head_linear
        )

    else:
        raise ValueError(f"Unknown architecture: {arch}")

    return model.to(DEVICE)



# Verify new freeze ratios
print("=== REVISED FREEZE AUDIT ===\n")
for arch in ARCHITECTURES:
    weights = get_weights_enum(arch)
    model   = get_model(arch=arch, weights=weights)
    dummy   = torch.zeros(2, 1, 224, 224).to(DEVICE)
    with torch.no_grad():
        out = model(dummy)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    frozen    = total - trainable
    print(f"  {arch:<15} pretrained  "
          f"output={list(out.shape)}  "a
          f"trainable={trainable/1e6:.2f}M  "
          f"frozen={frozen/1e6:.2f}M  "
          f"({frozen/total*100:.1f}% frozen)")
    del model, dummy, out
    torch.cuda.empty_cache()

print("✓ Model factory updated (dropout=0.5)")

=== REVISED FREEZE AUDIT ===

  resnet18        pretrained  output=[2, 11]  trainable=8.40M  frozen=2.78M  (24.8% frozen)
  densenet121     pretrained  output=[2, 11]  trainable=5.01M  frozen=1.95M  (28.0% frozen)
✓ Model factory updated (dropout=0.5)


In [5]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def compute_pos_weights(labels_mh: np.ndarray) -> torch.Tensor:
    """
    Compute per-class positive weights for BCEWithLogitsLoss.
    Formula: pos_weight[c] = (N - N_c) / N_c  (inverse frequency)
    Clamped to [0.1, 20] to avoid extreme values.

    Args:
        labels_mh : (N, NUM_CLASSES) float32 multi-hot

    Returns:
        pos_weights : (NUM_CLASSES,) float32 tensor
    """
    n          = len(labels_mh)
    pos_counts = labels_mh.sum(axis=0).clip(min=1)          # (NUM_CLASSES,)
    neg_counts = n - pos_counts
    weights    = neg_counts / pos_counts
    weights    = np.clip(weights, 0.1, 20.0)
    return torch.tensor(weights, dtype=torch.float32)


def aggregate_subject_probs(
    subject_ids : np.ndarray,    # (N,)
    probs       : np.ndarray,    # (N, NUM_CLASSES) sigmoid outputs
    labels_mh   : np.ndarray,    # (N, NUM_CLASSES) multi-hot float32
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Aggregate image-level predictions to subject-level by mean pooling.

    Returns:
        subj_ids    : (M,)              int64
        subj_probs  : (M, NUM_CLASSES)  float32  mean sigmoid per class
        subj_labels : (M, NUM_CLASSES)  float32  majority-vote multi-hot
    """
    subj_to_probs  = {}
    subj_to_labels = {}

    for sid, p, y in zip(subject_ids, probs, labels_mh):
        sid = int(sid)
        subj_to_probs.setdefault(sid, []).append(p)
        subj_to_labels.setdefault(sid, []).append(y)

    subj_ids = np.array(list(subj_to_probs.keys()), dtype=np.int64)

    subj_probs = np.array(
        [np.mean(subj_to_probs[sid], axis=0) for sid in subj_ids],
        dtype=np.float32
    )   # (M, NUM_CLASSES)

    # Majority vote per class across all images for this subject
    subj_labels = np.array(
        [(np.mean(subj_to_labels[sid], axis=0) >= 0.5).astype(np.float32)
         for sid in subj_ids],
        dtype=np.float32
    )   # (M, NUM_CLASSES)

    return subj_ids, subj_probs, subj_labels


def compute_metrics(
    subj_probs  : np.ndarray,             # (M, NUM_CLASSES) sigmoid probs
    subj_labels : np.ndarray,             # (M, NUM_CLASSES) multi-hot
    genders     : Optional[np.ndarray] = None,  # (M,) int8
) -> Dict:
    """
    Compute multi-label evaluation metrics.

    Primary: macro-averaged per-class binary AUC
    Also: per-class AUC, balanced accuracy (threshold=0.5), top-1 accuracy
    Optionally: all metrics stratified by gender
    """
    results = {}
    preds   = (subj_probs >= 0.5).astype(np.float32)   # (M, NUM_CLASSES)

    # ── Per-class binary AUC ──────────────────────────────────────────────────
    per_class_auc = {}
    auc_vals      = []

    for c in range(NUM_CLASSES):
        cls_name = SHORT_LIST[c]
        y_true   = subj_labels[:, c]
        y_score  = subj_probs[:, c]

        if y_true.sum() < 2 or (1 - y_true).sum() < 2:
            per_class_auc[cls_name] = float('nan')
            continue
        try:
            auc = float(roc_auc_score(y_true, y_score))
            per_class_auc[cls_name] = auc
            auc_vals.append(auc)
        except ValueError:
            per_class_auc[cls_name] = float('nan')

    macro_auc = float(np.mean(auc_vals)) if auc_vals else float('nan')
    results['macro_auc']     = macro_auc
    results['per_class_auc'] = per_class_auc

    # ── Threshold-based metrics (argmax on probs → int label for bal_acc) ─────
    # Use argmax for balanced accuracy — single predicted class per image
    pred_int  = subj_probs.argmax(axis=1)
    true_int  = subj_labels.argmax(axis=1)   # safe since single-label dataset
    results['bal_acc']  = float(balanced_accuracy_score(true_int, pred_int))
    results['top1_acc'] = float((pred_int == true_int).mean())

    # ── Gender-stratified metrics ─────────────────────────────────────────────
    if genders is not None:
        for gender_val, gender_name in [(1, 'male'), (0, 'female')]:
            mask = (genders == gender_val)

            if mask.sum() < 10:
                results[f'{gender_name}_macro_auc']     = float('nan')
                results[f'{gender_name}_bal_acc']       = float('nan')
                results[f'{gender_name}_n']             = int(mask.sum())
                results[f'{gender_name}_per_class_auc'] = {}
                continue

            g_probs  = subj_probs[mask]
            g_labels = subj_labels[mask]

            g_per_class = {}
            g_auc_vals  = []

            for c in range(NUM_CLASSES):
                cls_name = SHORT_LIST[c]
                y_true   = g_labels[:, c]
                y_score  = g_probs[:, c]

                if y_true.sum() < 2 or (1 - y_true).sum() < 2:
                    g_per_class[cls_name] = float('nan')
                    continue
                try:
                    auc = float(roc_auc_score(y_true, y_score))
                    g_per_class[cls_name] = auc
                    g_auc_vals.append(auc)
                except ValueError:
                    g_per_class[cls_name] = float('nan')

            g_pred_int = g_probs.argmax(axis=1)
            g_true_int = g_labels.argmax(axis=1)

            results[f'{gender_name}_macro_auc']     = float(
                np.mean(g_auc_vals) if g_auc_vals else float('nan')
            )
            results[f'{gender_name}_bal_acc']       = float(
                balanced_accuracy_score(g_true_int, g_pred_int)
            )
            results[f'{gender_name}_n']             = int(mask.sum())
            results[f'{gender_name}_per_class_auc'] = g_per_class

    return results


def check_completion(
    arch          : str,
    pretrained    : bool,
    condition     : str,
    attacker_type : str,
    eval_split    : str = 'val',
) -> bool:
    """Check if a run already exists in PER_RUN_CSV."""
    if not PER_RUN_CSV.exists():
        return False
    df       = pd.read_csv(PER_RUN_CSV)
    init_str = 'pretrained' if pretrained else 'scratch'
    mask     = (
        (df['arch']          == arch)          &
        (df['init']          == init_str)      &
        (df['train_cond']    == condition)     &
        (df['attacker_type'] == attacker_type) &
        (df['eval_split']    == eval_split)
    )
    return bool(mask.any())


def log_result(row: Dict) -> None:
    """Append one result row to PER_RUN_CSV."""
    df_new = pd.DataFrame([row])
    if PER_RUN_CSV.exists():
        df_new.to_csv(PER_RUN_CSV, mode='a', header=False, index=False)
    else:
        df_new.to_csv(PER_RUN_CSV, index=False)


def get_results_summary() -> Optional[pd.DataFrame]:
    """Load and display current results summary."""
    if not PER_RUN_CSV.exists():
        print("No results yet.")
        return None
    df      = pd.read_csv(PER_RUN_CSV)
    summary = df.groupby(
        ['arch', 'init', 'train_cond', 'attacker_type']
    )['macro_auc'].agg(['mean', 'std']).round(4)
    return summary


# ── Smoke tests ───────────────────────────────────────────────────────────────
print("=== UTILITY FUNCTION SMOKE TESTS ===\n")

# pos weights
pw = compute_pos_weights(TRAIN_LABELS_MH)
print("Positive weights (train):")
for c in range(NUM_CLASSES):
    n_pos = int(TRAIN_LABELS_MH[:, c].sum())
    print(f"  {c:2d} {SHORT_LIST[c]:<10}  "
          f"pos_weight={pw[c]:.3f}  n_pos={n_pos:,}")

# aggregate_subject_probs
dummy_probs  = np.random.rand(100, NUM_CLASSES).astype(np.float32)
dummy_probs /= dummy_probs.sum(axis=1, keepdims=True)   # normalise rows
dummy_sids   = TRAIN_SIDS[:100]
dummy_mh     = TRAIN_LABELS_MH[:100]

s_ids, s_probs, s_labels = aggregate_subject_probs(
    dummy_sids, dummy_probs, dummy_mh
)
print(f"\naggregate_subject_probs:")
print(f"  Input  : {len(dummy_sids)} images, "
      f"{len(np.unique(dummy_sids))} subjects")
print(f"  Output : {len(s_ids)} subjects  "
      f"probs={s_probs.shape}  labels={s_labels.shape}")
print(f"  Labels sum/row (should be ~1.0): "
      f"{s_labels.sum(axis=1).mean():.2f}")

# compute_metrics
sid_to_gender = {int(s): int(g)
                 for s, g in zip(TRAIN_SIDS, TRAIN_GENDERS)}
s_genders = np.array(
    [sid_to_gender[sid] for sid in s_ids], dtype=np.int8
)

metrics = compute_metrics(s_probs, s_labels, genders=s_genders)
print(f"\ncompute_metrics (random probs, sanity only):")
print(f"  macro_auc        : {metrics['macro_auc']:.4f}")
print(f"  bal_acc          : {metrics['bal_acc']:.4f}")
print(f"  male_macro_auc   : {metrics.get('male_macro_auc', 'n/a')}")
print(f"  female_macro_auc : {metrics.get('female_macro_auc', 'n/a')}")

print("\n✓ Utility functions ready")

=== UTILITY FUNCTION SMOKE TESTS ===

Positive weights (train):
   0 NoFind      pos_weight=4.766  n_pos=4,276
   1 Infilt      pos_weight=2.701  n_pos=6,661
   2 Atelec      pos_weight=7.343  n_pos=2,955
   3 Effus       pos_weight=7.888  n_pos=2,774
   4 Nodule      pos_weight=11.867  n_pos=1,916
   5 PneuTx      pos_weight=15.647  n_pos=1,481
   6 Mass        pos_weight=15.360  n_pos=1,507
   7 Consol      pos_weight=20.000  n_pos=944
   8 PlThck      pos_weight=20.000  n_pos=776
   9 CardMeg     pos_weight=20.000  n_pos=739
  10 Emphy       pos_weight=20.000  n_pos=625

aggregate_subject_probs:
  Input  : 100 images, 99 subjects
  Output : 99 subjects  probs=(99, 11)  labels=(99, 11)
  Labels sum/row (should be ~1.0): 1.01

compute_metrics (random probs, sanity only):
  macro_auc        : 0.4858
  bal_acc          : 0.0699
  male_macro_auc   : 0.5183412496430638
  female_macro_auc : 0.49854016160942155

✓ Utility functions ready


In [8]:
# =============================================================================
# GS TENSOR BUILDER
# =============================================================================

from gs_functions import GS_batch_image


def tensor_path(split: str, condition: str) -> Path:
    return TENSOR_DIR / f'{split}_{condition}.pt'


def build_and_save_tensor(
    split     : str,
    condition : str,
    images    : np.ndarray,
    labels_int: np.ndarray,
    labels_mh : np.ndarray,
    sids      : np.ndarray,
    genders   : np.ndarray,
) -> None:
    """Build tensors for one split/condition and save to disk."""
    out_path = tensor_path(split, condition)

    if out_path.exists():
        size_gb = out_path.stat().st_size / 1e9
        print(f"  ✓ EXISTS  {out_path.name:<35} ({size_gb:.2f} GB) — skip")
        return

    print(f"  Building {split}_{condition}  ({len(images):,} images)...")
    t0 = time.time()

    X = images.copy()

    if condition == 'raw':
        pass

    elif condition.startswith('gs'):
        mask_pct = int(condition.replace('gs', '')) / 100.0
        print(f"    Applying GS (maskP={mask_pct}, "
              f"ite={GS_ITERATIONS}, batch={GS_BATCH_SIZE})...")
        X = GS_batch_image(
            X,
            batch_size = GS_BATCH_SIZE,
            ite        = GS_ITERATIONS,
            maskP      = mask_pct,
        ).astype(np.float32)
        print(f"    GS done — range=[{X.min():.3f}, {X.max():.3f}]")
    else:
        raise ValueError(f"Unknown condition: {condition}")

    X_t    = torch.from_numpy(X).unsqueeze(1).float()      # (N, 1, 224, 224)
    y_int  = torch.from_numpy(labels_int).long()           # (N,)
    y_mh   = torch.from_numpy(labels_mh).float()           # (N, NUM_CLASSES)

    torch.save(
        {
            'x'         : X_t,
            'y_int'     : y_int,
            'y_mh'      : y_mh,
            'sids'      : sids,
            'genders'   : genders,
        },
        out_path
    )

    elapsed = time.time() - t0
    size_gb = out_path.stat().st_size / 1e9
    print(f"    Saved → {out_path.name}  "
          f"({size_gb:.2f} GB, {elapsed/60:.1f} min)")

    del X, X_t, y_int, y_mh
    gc.collect()


def load_tensors(
    split     : str,
    condition : str,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, np.ndarray, np.ndarray]:
    """
    Load a pre-built split/condition tensor from disk.
    Handles both old format (y) and new format (y_int, y_mh).

    Returns:
        X       : (N, 1, 224, 224) float32 tensor
        y_int   : (N,)             long tensor
        y_mh    : (N, NUM_CLASSES) float32 tensor
        sids    : (N,)             int64 numpy
        genders : (N,)             int8 numpy
    """
    path = tensor_path(split, condition)
    if not path.exists():
        raise FileNotFoundError(
            f"Tensor not found: {path}\n"
            f"Run pre-transformation cell first."
        )
    obj = torch.load(path, map_location='cpu', weights_only=False)

    X       = obj['x']
    sids    = obj['sids']
    genders = obj['genders']

    # ── Handle old format (y = long tensor) vs new format (y_int + y_mh) ─────
    if 'y_int' in obj:
        y_int = obj['y_int']
        y_mh  = obj['y_mh']
    else:
        # Old format — y is integer labels, reconstruct multi-hot on the fly
        y_int  = obj['y']
        y_np   = y_int.numpy()
        y_mh_np = np.zeros((len(y_np), NUM_CLASSES), dtype=np.float32)
        y_mh_np[np.arange(len(y_np)), y_np] = 1.0
        y_mh   = torch.from_numpy(y_mh_np)

    return X, y_int, y_mh, sids, genders


print("✓ load_tensors patched — handles old and new tensor formats")


# =============================================================================
# SPLIT × CONDITION MATRIX — test now includes all GS conditions
# =============================================================================
SPLITS_TO_BUILD = {
    'train': (TRAIN_IMAGES, TRAIN_LABELS, TRAIN_LABELS_MH,
              TRAIN_SIDS,   TRAIN_GENDERS),
    'val'  : (VAL_IMAGES,   VAL_LABELS,   VAL_LABELS_MH,
              VAL_SIDS,     VAL_GENDERS),
    'test' : (TEST_IMAGES,  TEST_LABELS,  TEST_LABELS_MH,
              TEST_SIDS,    TEST_GENDERS),
}

SPLIT_CONDITIONS = {
    'train': CONDITIONS,           # raw + all GS
    'val'  : CONDITIONS,           # raw + all GS
    'test' : CONDITIONS,           # raw + all GS — needed for adaptive test eval
}

# =============================================================================
# PRE-TRANSFORMATION — build all tensors upfront
# =============================================================================
print("=" * 70)
print("PRE-TRANSFORMATION — building all condition tensors")
print("=" * 70)
print(f"Output dir : {TENSOR_DIR}")
print(f"Conditions : {CONDITIONS}\n")

total_t0 = time.time()

for split, (images, labels_int, labels_mh,
            sids, genders) in SPLITS_TO_BUILD.items():
    conds = SPLIT_CONDITIONS[split]
    print(f"\n── {split.upper()}  "
          f"({len(images):,} images × {len(conds)} conditions) ──")
    for condition in conds:
        build_and_save_tensor(
            split      = split,
            condition  = condition,
            images     = images,
            labels_int = labels_int,
            labels_mh  = labels_mh,
            sids       = sids,
            genders    = genders,
        )

total_elapsed = (time.time() - total_t0) / 60
print(f"\n{'='*70}")
print(f"PRE-TRANSFORMATION COMPLETE — {total_elapsed:.1f} min total")
print(f"{'='*70}")

# Disk usage summary
print(f"\n=== TENSOR FILES ON DISK ===")
total_size = 0
for f in sorted(TENSOR_DIR.iterdir()):
    size_gb = f.stat().st_size / 1e9
    total_size += size_gb
    print(f"  {f.name:<35} {size_gb:.2f} GB")
print(f"\n  Total : {total_size:.2f} GB")

# Pre-load test_raw into memory for blind eval
print(f"\n=== LOADING test_raw INTO MEMORY (blind eval anchor) ===")
TEST_X_RAW, TEST_Y_INT_RAW, TEST_Y_MH_RAW, \
    TEST_SIDS_RAW, TEST_GENDERS_RAW = load_tensors('test', 'raw')
print(f"  test_raw: {TEST_X_RAW.shape}  "
      f"({TEST_X_RAW.element_size() * TEST_X_RAW.nelement() / 1e9:.2f} GB)")

print("\n✓ Tensor builder ready")

✓ load_tensors patched — handles old and new tensor formats
PRE-TRANSFORMATION — building all condition tensors
Output dir : cache/nih_cxr/utility/tensors
Conditions : ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']


── TRAIN  (24,654 images × 7 conditions) ──
  ✓ EXISTS  train_raw.pt                        (4.95 GB) — skip
  ✓ EXISTS  train_gs50.pt                       (4.95 GB) — skip
  ✓ EXISTS  train_gs40.pt                       (4.95 GB) — skip
  ✓ EXISTS  train_gs30.pt                       (4.95 GB) — skip
  ✓ EXISTS  train_gs20.pt                       (4.95 GB) — skip
  ✓ EXISTS  train_gs10.pt                       (4.95 GB) — skip
  ✓ EXISTS  train_gs0.pt                        (4.95 GB) — skip

── VAL  (3,725 images × 7 conditions) ──
  ✓ EXISTS  val_raw.pt                          (0.75 GB) — skip
  ✓ EXISTS  val_gs50.pt                         (0.75 GB) — skip
  ✓ EXISTS  val_gs40.pt                         (0.75 GB) — skip
  ✓ EXISTS  val_gs30.pt                

In [9]:
from torchvision import transforms

# ── Augmentation ──────────────────────────────────────────────────────────────
TRAIN_AUGMENT = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.2, contrast=0.2)
    ], p=0.5),
])

def augment_batch(imgs: torch.Tensor) -> torch.Tensor:
    return torch.stack([TRAIN_AUGMENT(img) for img in imgs])


# ── Optimizer ─────────────────────────────────────────────────────────────────
def get_optimizer(
    model      : nn.Module,
    arch       : str,
    pretrained : bool,
    lr         : float,
    wd         : float,
) -> optim.Optimizer:
    if not pretrained:
        return optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=lr, weight_decay=wd
        )
    if arch == 'resnet18':
        head_names = ['fc']
    else:
        head_names = ['classifier']

    head_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and
                   any(n.startswith(s) for s in head_names)]
    body_params = [p for n, p in model.named_parameters()
                   if p.requires_grad and
                   not any(n.startswith(s) for s in head_names)]

    return optim.Adam([
        {'params': body_params,
         'lr': lr / HEAD_LR_MULTIPLIER, 'weight_decay': wd},
        {'params': head_params,
         'lr': lr,                      'weight_decay': wd},
    ])


# ── Train & Evaluate ──────────────────────────────────────────────────────────
def train_and_evaluate(
    arch           : str,
    pretrained     : bool,
    condition      : str,
    train_x        : torch.Tensor,          # (N, 1, 224, 224)
    train_y_mh     : torch.Tensor,          # (N, NUM_CLASSES) float32
    train_sids     : np.ndarray,
    train_genders  : np.ndarray,
    val_x          : torch.Tensor,
    val_y_mh       : torch.Tensor,          # (N, NUM_CLASSES) float32
    val_sids       : np.ndarray,
    val_genders    : np.ndarray,
    attacker_type  : str = 'adaptive',
    eval_condition : Optional[str] = None,
    eval_split     : str = 'val',
) -> None:

    if eval_condition is None:
        eval_condition = condition

    init_str     = 'pretrained' if pretrained else 'scratch'
    setting_name = (f"{arch}_{init_str}_"
                    f"train-{condition}_"
                    f"eval-{eval_condition}_"
                    f"{attacker_type}_{eval_split}")

    ckpt_dir  = MODELS_DIR / arch / init_str / condition
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = ckpt_dir / 'best.pt'

    lr = LR_PRETRAINED if pretrained else LR_SCRATCH
    wd = WD_PRETRAINED if pretrained else WD_SCRATCH

    print("\n" + "=" * 80)
    print(f"▶  {setting_name}")
    if pretrained:
        print(f"   LR head={lr:.1e}  body={lr/HEAD_LR_MULTIPLIER:.1e}  "
              f"WD={wd:.1e}  dropout={DROPOUT_P}")
    else:
        print(f"   LR={lr:.1e}  WD={wd:.1e}  dropout={DROPOUT_P}")
    print(f"   train={len(train_x):,}  val={len(val_x):,}  "
          f"task=multi-label BCE")
    ckpt_exists = ckpt_path.exists()
    if ckpt_exists:
        print(f"   ✓ Checkpoint exists → SKIP TRAIN, EVAL ONLY")
    print("=" * 80)

    weights = get_weights_enum(arch) if pretrained else None
    model   = get_model(arch=arch, weights=weights)

    # ── Loss: BCEWithLogitsLoss with per-class pos_weight ─────────────────────
    pos_weights     = compute_pos_weights(
        train_y_mh.numpy()
    ).to(DEVICE)

    criterion_train = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
    criterion_val   = nn.BCEWithLogitsLoss()   # no weighting for val loss

    optimizer = get_optimizer(model, arch, pretrained, lr, wd)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=4
    )

    # ── DataLoaders ───────────────────────────────────────────────────────────
    train_loader = DataLoader(
        TensorDataset(train_x, train_y_mh),
        batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, pin_memory=True
    )
    val_loader = DataLoader(
        TensorDataset(val_x, val_y_mh),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, pin_memory=True
    )

    best_val_auc      = float('nan')
    best_epoch        = float('nan')
    best_val_loss     = float('nan')

    # ── Training ──────────────────────────────────────────────────────────────
    if not ckpt_exists:
        best_val_auc      = -1.0
        best_epoch        = -1
        best_val_loss     = float('inf')
        epochs_no_improve = 0
        use_amp           = (DEVICE.type == 'cuda')
        scaler            = torch.cuda.amp.GradScaler(enabled=use_amp)

        for epoch in range(NUM_EPOCHS):

            # Train
            model.train()
            running_loss = 0.0
            for imgs, y_mh in train_loader:
                imgs = augment_batch(imgs)
                imgs, y_mh = imgs.to(DEVICE), y_mh.to(DEVICE)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type='cuda',
                                        enabled=use_amp):
                    logits = model(imgs)               # (B, NUM_CLASSES)
                    loss   = criterion_train(logits, y_mh)

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(), max_norm=1.0
                )
                scaler.step(optimizer)
                scaler.update()
                running_loss += float(loss.item())

            train_loss = running_loss / max(len(train_loader), 1)

            # Validate
            model.eval()
            val_loss_sum = 0.0
            all_probs    = []
            all_y_mh     = []

            with torch.no_grad():
                for imgs, y_mh in val_loader:
                    imgs, y_mh = imgs.to(DEVICE), y_mh.to(DEVICE)
                    logits = model(imgs)
                    val_loss_sum += float(
                        criterion_val(logits, y_mh).item()
                    )
                    # sigmoid → per-class probabilities
                    all_probs.append(
                        torch.sigmoid(logits).cpu().numpy()
                    )
                    all_y_mh.append(y_mh.cpu().numpy())

            val_loss  = val_loss_sum / max(len(val_loader), 1)
            all_probs = np.concatenate(all_probs,  axis=0)  # (N, C)
            all_y_mh  = np.concatenate(all_y_mh,   axis=0)  # (N, C)

            _, s_probs, s_labels = aggregate_subject_probs(
                val_sids, all_probs, all_y_mh
            )

            # Per-class binary AUC → macro average
            auc_vals = []
            for c in range(NUM_CLASSES):
                y_t = s_labels[:, c]
                if y_t.sum() < 2 or (1 - y_t).sum() < 2:
                    continue
                try:
                    auc_vals.append(
                        roc_auc_score(y_t, s_probs[:, c])
                    )
                except ValueError:
                    pass
            subj_auc = float(np.mean(auc_vals)) if auc_vals \
                else float('nan')

            subj_bacc = balanced_accuracy_score(
                s_labels.argmax(axis=1),
                s_probs.argmax(axis=1)
            )

            scheduler.step(
                subj_auc if not np.isnan(subj_auc) else 0.0
            )
            cur_lr = optimizer.param_groups[-1]['lr']

            print(f"  Epoch {epoch+1:02d} | "
                  f"trainLoss={train_loss:.4f} | "
                  f"valLoss={val_loss:.4f} | "
                  f"valAUC={subj_auc:.4f} | "
                  f"valBAcc={subj_bacc:.4f} | "
                  f"lr={cur_lr:.2e}")

            if not np.isnan(subj_auc) and subj_auc > best_val_auc:
                best_val_auc      = float(subj_auc)
                best_epoch        = int(epoch + 1)
                best_val_loss     = float(val_loss)
                epochs_no_improve = 0
                torch.save(model.state_dict(), ckpt_path)
                print(f"  💾 Saved  (AUC={best_val_auc:.4f})")
            else:
                epochs_no_improve += 1

            if epochs_no_improve >= PATIENCE:
                print(f"  🛑 Early stop @ epoch {epoch+1} "
                      f"(best: epoch {best_epoch}, "
                      f"AUC={best_val_auc:.4f})")
                break

    # ── Evaluation helper ─────────────────────────────────────────────────────
    def run_eval(
        loader      : DataLoader,
        sids        : np.ndarray,
        genders     : np.ndarray,
        split_name  : str,
        atk_type    : str,
        eval_cond   : str,
    ) -> None:
        model.eval()
        all_probs = []
        all_y_mh  = []
        with torch.no_grad():
            for imgs, y_mh in loader:
                imgs = imgs.to(DEVICE)
                all_probs.append(
                    torch.sigmoid(model(imgs)).cpu().numpy()
                )
                all_y_mh.append(y_mh.numpy())

        all_probs = np.concatenate(all_probs, axis=0)
        all_y_mh  = np.concatenate(all_y_mh,  axis=0)

        s_ids, s_probs, s_labels = aggregate_subject_probs(
            sids, all_probs, all_y_mh
        )
        sid_to_g = {int(s): int(g) for s, g in zip(sids, genders)}
        s_genders = np.array(
            [sid_to_g[sid] for sid in s_ids], dtype=np.int8
        )

        metrics = compute_metrics(s_probs, s_labels, genders=s_genders)

        row = {
            'arch'            : arch,
            'init'            : init_str,
            'train_cond'      : condition,
            'eval_cond'       : eval_cond,
            'attacker_type'   : atk_type,
            'eval_split'      : split_name,
            'best_epoch'      : best_epoch,
            'best_val_auc'    : best_val_auc,
            'macro_auc'       : metrics['macro_auc'],
            'bal_acc'         : metrics['bal_acc'],
            'top1_acc'        : metrics['top1_acc'],
            'male_macro_auc'  : metrics.get('male_macro_auc',   float('nan')),
            'female_macro_auc': metrics.get('female_macro_auc', float('nan')),
            'male_n'          : metrics.get('male_n',           0),
            'female_n'        : metrics.get('female_n',         0),
            'male_bal_acc'    : metrics.get('male_bal_acc',     float('nan')),
            'female_bal_acc'  : metrics.get('female_bal_acc',   float('nan')),
        }
        for cls_short, v in metrics.get('per_class_auc', {}).items():
            row[f'auc_{cls_short}'] = v
        for g in ['male', 'female']:
            for cls_short, v in metrics.get(
                f'{g}_per_class_auc', {}
            ).items():
                row[f'{g}_auc_{cls_short}'] = v

        log_result(row)

        print(f"\n  ── {split_name.upper()} | {atk_type} ──")
        print(f"  macro_AUC    : {metrics['macro_auc']:.4f}")
        print(f"  bal_acc      : {metrics['bal_acc']:.4f}")
        print(f"  top1_acc     : {metrics['top1_acc']:.4f}")
        print(f"  male_AUC     : "
              f"{metrics.get('male_macro_auc', float('nan')):.4f}"
              f"  (n={metrics.get('male_n', 0)})")
        print(f"  female_AUC   : "
              f"{metrics.get('female_macro_auc', float('nan')):.4f}"
              f"  (n={metrics.get('female_n', 0)})")
        print(f"  Per-class AUC:")
        for cls_short, v in metrics.get('per_class_auc', {}).items():
            print(f"    {cls_short:<10} {v:.4f}")

    # ── Load best checkpoint ──────────────────────────────────────────────────
    model.load_state_dict(
        torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
    )

    # ── Val evaluation (adaptive) ─────────────────────────────────────────────
    run_eval(
        loader     = val_loader,
        sids       = val_sids,
        genders    = val_genders,
        split_name = eval_split,
        atk_type   = attacker_type,
        eval_cond  = eval_condition,
    )

    # ── Adaptive test evaluation ──────────────────────────────────────────────
    # Only for adaptive runs — load matching test condition tensor
    if attacker_type == 'adaptive':
        try:
            test_x_cond, _, test_y_mh_cond, \
                test_sids_cond, test_genders_cond = load_tensors(
                    'test', condition
                )
            test_loader_cond = DataLoader(
                TensorDataset(test_x_cond, test_y_mh_cond),
                batch_size=BATCH_SIZE, shuffle=False,
                num_workers=4, pin_memory=True
            )
            run_eval(
                loader     = test_loader_cond,
                sids       = test_sids_cond,
                genders    = test_genders_cond,
                split_name = 'test',
                atk_type   = 'adaptive',
                eval_cond  = condition,
            )
            del test_x_cond, test_y_mh_cond
            torch.cuda.empty_cache()
        except FileNotFoundError:
            print(f"  ⚠ test_{condition}.pt not found — skipping test eval")

    del model
    torch.cuda.empty_cache()
    gc.collect()


print("✓ train_and_evaluate ready (multi-label BCE)")
print(f"  Loss           : BCEWithLogitsLoss + per-class pos_weight")
print(f"  Inference      : sigmoid (per-class independent)")
print(f"  Primary metric : macro-averaged per-class binary AUC")
print(f"  Augmentation   : flip + rotation±10° + brightness/contrast")
print(f"  Differential LR: head={LR_PRETRAINED:.1e}  "
      f"body={LR_PRETRAINED/HEAD_LR_MULTIPLIER:.1e}  (pretrained only)")
print(f"  Dropout        : {DROPOUT_P}")
print(f"  Patience       : {PATIENCE}  |  Max epochs : {NUM_EPOCHS}")

✓ train_and_evaluate ready (multi-label BCE)
  Loss           : BCEWithLogitsLoss + per-class pos_weight
  Inference      : sigmoid (per-class independent)
  Primary metric : macro-averaged per-class binary AUC
  Augmentation   : flip + rotation±10° + brightness/contrast
  Differential LR: head=5.0e-05  body=1.0e-05  (pretrained only)
  Dropout        : 0.5
  Patience       : 10  |  Max epochs : 100


## Pre Transform for GS Variants
- This is a one time execution.
- Sequential runs will skipped as long as all variants are exist in the target directory.

In [10]:
# =============================================================================
# PRE-TRANSFORMATION — build all condition tensors upfront
# =============================================================================
# Saves train/val for all 7 conditions + test_raw to disk.
# Total ~42 GB. Only runs once — skips already-built files.
# Loop order: condition → models (never recompute same GS twice)
# =============================================================================

import os

TENSOR_DIR = CACHE_DIR / 'tensors'
TENSOR_DIR.mkdir(parents=True, exist_ok=True)

# test_raw is kept in memory throughout for blind evaluation
# All other tensors are loaded one condition at a time during training

SPLITS_TO_BUILD = {
    'train' : (TRAIN_IMAGES, TRAIN_LABELS, TRAIN_SIDS, TRAIN_GENDERS),
    'val'   : (VAL_IMAGES,   VAL_LABELS,   VAL_SIDS,   VAL_GENDERS),
    'test'  : (TEST_IMAGES,  TEST_LABELS,  TEST_SIDS,  TEST_GENDERS),
}

# For test split we only need raw (blind eval always uses raw test)
SPLIT_CONDITIONS = {
    'train' : CONDITIONS,           # all 7
    'val'   : CONDITIONS,           # all 7
    'test'  : ['raw'],              # raw only
}


def tensor_path(split: str, condition: str) -> Path:
    return TENSOR_DIR / f'{split}_{condition}.pt'


def build_and_save_tensor(
    split     : str,
    condition : str,
    images    : np.ndarray,
    labels    : np.ndarray,
    sids      : np.ndarray,
    genders   : np.ndarray,
) -> None:
    """Build tensors for one split/condition and save to disk."""
    out_path = tensor_path(split, condition)

    if out_path.exists():
        size_gb = out_path.stat().st_size / 1e9
        print(f"  ✓ EXISTS  {out_path.name:<30} ({size_gb:.2f} GB) — skip")
        return

    print(f"  Building {split}_{condition} "
          f"({len(images):,} images)...")
    t0 = time.time()

    X = images.copy()

    if condition == 'raw':
        pass

    elif condition.startswith('gs'):
        mask_pct = int(condition.replace('gs', '')) / 100.0
        print(f"    Applying GS (maskP={mask_pct}, "
              f"ite={GS_ITERATIONS}, batch={GS_BATCH_SIZE})...")
        X = GS_batch_image(
            X,
            batch_size = GS_BATCH_SIZE,
            ite        = GS_ITERATIONS,
            maskP      = mask_pct,
        ).astype(np.float32)
        print(f"    GS done — range=[{X.min():.3f}, {X.max():.3f}]")

    else:
        raise ValueError(f"Unknown condition: {condition}")

    X_t = torch.from_numpy(X).unsqueeze(1).float()  # (N, 1, 224, 224)
    y_t = torch.from_numpy(labels).long()

    torch.save(
        {'x': X_t, 'y': y_t, 'sids': sids, 'genders': genders},
        out_path
    )

    elapsed = time.time() - t0
    size_gb = out_path.stat().st_size / 1e9
    print(f"    Saved → {out_path.name}  "
          f"({size_gb:.2f} GB, {elapsed/60:.1f} min)")

    del X, X_t, y_t
    gc.collect()


def load_tensors(
    split     : str,
    condition : str,
) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray, np.ndarray]:
    """
    Load a pre-built split/condition tensor from disk.

    Returns:
        X       : (N, 1, 224, 224) float32 tensor
        y       : (N,) long tensor
        sids    : (N,) int64 numpy
        genders : (N,) int8 numpy
    """
    path = tensor_path(split, condition)
    if not path.exists():
        raise FileNotFoundError(
            f"Tensor not found: {path}\n"
            f"Run pre-transformation cell first."
        )
    obj = torch.load(path, map_location='cpu', weights_only=False)
    return obj['x'], obj['y'], obj['sids'], obj['genders']


# =============================================================================
# RUN PRE-TRANSFORMATION
# =============================================================================
print("=" * 70)
print("PRE-TRANSFORMATION — building all condition tensors")
print("=" * 70)
print(f"Output dir : {TENSOR_DIR}")
print(f"Conditions : {CONDITIONS}")
print()

total_t0 = time.time()

for split, (images, labels, sids, genders) in SPLITS_TO_BUILD.items():
    conditions_for_split = SPLIT_CONDITIONS[split]
    print(f"\n── {split.upper()} "
          f"({len(images):,} images × "
          f"{len(conditions_for_split)} conditions) ──")

    for condition in conditions_for_split:
        build_and_save_tensor(
            split=split, condition=condition,
            images=images, labels=labels,
            sids=sids, genders=genders,
        )

total_elapsed = (time.time() - total_t0) / 60
print(f"\n{'='*70}")
print(f"PRE-TRANSFORMATION COMPLETE — {total_elapsed:.1f} min total")
print(f"{'='*70}")

# Disk usage summary
print(f"\n=== TENSOR FILES ON DISK ===")
total_size = 0
for f in sorted(TENSOR_DIR.iterdir()):
    size_gb = f.stat().st_size / 1e9
    total_size += size_gb
    print(f"  {f.name:<30} {size_gb:.2f} GB")
print(f"\n  Total : {total_size:.2f} GB")

# Pre-load test_raw into memory — stays there for all blind evals
print(f"\n=== LOADING test_raw INTO MEMORY (blind eval anchor) ===")
TEST_X_RAW, TEST_Y_RAW, TEST_SIDS_RAW, TEST_GENDERS_RAW = load_tensors(
    'test', 'raw'
)
print(f"  test_raw loaded: {TEST_X_RAW.shape}  "
      f"({TEST_X_RAW.element_size() * TEST_X_RAW.nelement() / 1e9:.2f} GB)")
print(f"\n✓ Ready for experiment loop")

PRE-TRANSFORMATION — building all condition tensors
Output dir : cache/nih_cxr/utility/tensors
Conditions : ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']


── TRAIN (24,654 images × 7 conditions) ──
  ✓ EXISTS  train_raw.pt                   (4.95 GB) — skip
  ✓ EXISTS  train_gs50.pt                  (4.95 GB) — skip
  ✓ EXISTS  train_gs40.pt                  (4.95 GB) — skip
  ✓ EXISTS  train_gs30.pt                  (4.95 GB) — skip
  ✓ EXISTS  train_gs20.pt                  (4.95 GB) — skip
  ✓ EXISTS  train_gs10.pt                  (4.95 GB) — skip
  ✓ EXISTS  train_gs0.pt                   (4.95 GB) — skip

── VAL (3,725 images × 7 conditions) ──
  ✓ EXISTS  val_raw.pt                     (0.75 GB) — skip
  ✓ EXISTS  val_gs50.pt                    (0.75 GB) — skip
  ✓ EXISTS  val_gs40.pt                    (0.75 GB) — skip
  ✓ EXISTS  val_gs30.pt                    (0.75 GB) — skip
  ✓ EXISTS  val_gs20.pt                    (0.75 GB) — skip
  ✓ EXISTS  val_gs10.pt        

## CAUTION
- Run cell below only if you want a fresh start to a model training.
- Current pipeline is designed to resume where it was interrupt/crashed.
- If you run the cell below, you will lose all model checkpoints saved previously.

    - To do that, simply uncomment the cell below and run it once...

In [7]:
# import shutil

# print("=== CLEANING PREVIOUS RUNS ===\n")

# # Remove model checkpoints
# if MODELS_DIR.exists():
#     shutil.rmtree(MODELS_DIR)
#     MODELS_DIR.mkdir(parents=True, exist_ok=True)
#     print(f"✓ Cleared {MODELS_DIR}")

# # Remove results CSV
# if PER_RUN_CSV.exists():
#     PER_RUN_CSV.unlink()
#     print(f"✓ Removed {PER_RUN_CSV}")

# if SUMMARY_CSV.exists():
#     SUMMARY_CSV.unlink()
#     print(f"✓ Removed {SUMMARY_CSV}")

# # Verify tensors are still intact
# print(f"\n=== TENSOR FILES (should be untouched) ===")
# for f in sorted(TENSOR_DIR.iterdir()):
#     size_gb = f.stat().st_size / 1e9
#     print(f"  {f.name:<30} {size_gb:.2f} GB")

# print("\n✓ Ready for fresh run")

=== CLEANING PREVIOUS RUNS ===

✓ Cleared models/nih_cxr/utility
✓ Removed results/nih_cxr/utility/per_run_results.csv

=== TENSOR FILES (should be untouched) ===


NameError: name 'TENSOR_DIR' is not defined

## Main Experiment Loop

In [12]:
def load_tensors(
    split     : str,
    condition : str,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, np.ndarray, np.ndarray]:
    path = tensor_path(split, condition)
    if not path.exists():
        raise FileNotFoundError(
            f"Tensor not found: {path}\n"
            f"Run pre-transformation cell first."
        )
    obj = torch.load(path, map_location='cpu', weights_only=False)

    X       = obj['x']
    sids    = obj['sids']
    genders = obj['genders']

    if 'y_int' in obj:
        y_int = obj['y_int']
        y_mh  = obj['y_mh']
    else:
        y_int   = obj['y']
        y_np    = y_int.numpy()
        y_mh_np = np.zeros((len(y_np), NUM_CLASSES), dtype=np.float32)
        y_mh_np[np.arange(len(y_np)), y_np] = 1.0
        y_mh    = torch.from_numpy(y_mh_np)

    return X, y_int, y_mh, sids, genders

# Verify
X, y_int, y_mh, sids, genders = load_tensors('train', 'raw')
print(f"✓ load_tensors working")
print(f"  X     : {X.shape}")
print(f"  y_int : {y_int.shape}  dtype={y_int.dtype}")
print(f"  y_mh  : {y_mh.shape}  dtype={y_mh.dtype}  "
      f"sum/row={y_mh.sum(dim=1).mean():.1f}")
del X, y_int, y_mh, sids, genders

✓ load_tensors working
  X     : torch.Size([24654, 1, 224, 224])
  y_int : torch.Size([24654])  dtype=torch.int64
  y_mh  : torch.Size([24654, 11])  dtype=torch.float32  sum/row=1.0


In [ ]:
# =============================================================================
# MAIN EXPERIMENT LOOP
# =============================================================================
# Loop order: condition → pretrained first → arch
# Only one condition's train/val tensors in RAM at a time.
# test_raw stays in memory throughout for blind evaluation.
# =============================================================================

import subprocess

# =============================================================================
# PRE-FLIGHT CHECKS
# =============================================================================
print("=== PRE-FLIGHT CHECKS ===\n")

# 1. Verify all tensor files exist
all_ok = True
for split, conditions in SPLIT_CONDITIONS.items():
    for condition in conditions:
        p = tensor_path(split, condition)
        if not p.exists():
            print(f"  ✗ MISSING: {p.name}")
            all_ok = False
        else:
            size_gb = p.stat().st_size / 1e9
            print(f"  ✓ {p.name:<35} ({size_gb:.2f} GB)")

assert all_ok, "Some tensor files missing — re-run pre-transformation cell first"

# 2. Verify test_raw is in memory
assert 'TEST_X_RAW' in globals(), \
    "TEST_X_RAW not in memory — re-run end of tensor builder cell"
print(f"\n  test_raw in memory : {TEST_X_RAW.shape} ✓")

# 3. Show already-completed runs
if PER_RUN_CSV.exists():
    done_df = pd.read_csv(PER_RUN_CSV)
    print(f"\n  Completed runs : {len(done_df)}")
    print(done_df.groupby(
        ['arch', 'init', 'train_cond', 'attacker_type', 'eval_split']
    ).size().to_string())
else:
    print(f"\n  No completed runs yet — fresh start")

# 4. Disk space
result = subprocess.run('df -h .', shell=True,
                        capture_output=True, text=True)
print(f"\n{result.stdout.strip()}")
print("\n✓ Pre-flight complete — starting experiment loop")

# =============================================================================
# EXPERIMENT LOOP
# =============================================================================
print("\n" + "=" * 80)
print("STARTING NIH CXR UTILITY EXPERIMENTS")
print("=" * 80)
print(f"  Conditions    : {CONDITIONS}")
print(f"  Architectures : {ARCHITECTURES}")
print(f"  Init modes    : pretrained first, then scratch")
print(f"  Total runs    : "
      f"{len(CONDITIONS) * len(ARCHITECTURES) * 2} adaptive + "
      f"{(len(CONDITIONS)-1) * len(ARCHITECTURES) * 2} blind")
print(f"  Task          : multi-label BCE")
print("=" * 80)

for condition in CONDITIONS:
    print(f"\n{'#'*80}")
    print(f"# CONDITION: {condition}")
    print(f"{'#'*80}")

    # ── Load train & val tensors ──────────────────────────────────────────────
    print(f"\n  Loading train_{condition}...")
    train_x, train_y_int, train_y_mh, \
        train_sids, train_genders = load_tensors('train', condition)

    print(f"  Loading val_{condition}...")
    val_x, val_y_int, val_y_mh, \
        val_sids, val_genders = load_tensors('val', condition)

    print(f"  train: {train_x.shape}  val: {val_x.shape}")
    print(f"  train labels: {train_y_mh.shape}  "
          f"val labels: {val_y_mh.shape}")

    # ── Loop: pretrained first, then scratch ──────────────────────────────────
    for pretrained in [True, False]:
        init_str = 'pretrained' if pretrained else 'scratch'

        for arch in ARCHITECTURES:

            # ── A. ADAPTIVE ───────────────────────────────────────────────────
            if check_completion(arch, pretrained, condition,
                                attacker_type='adaptive',
                                eval_split='val'):
                print(f"\n⏩ Skip adaptive val: "
                      f"{arch} {init_str} {condition}")
            else:
                train_and_evaluate(
                    arch           = arch,
                    pretrained     = pretrained,
                    condition      = condition,
                    train_x        = train_x,
                    train_y_mh     = train_y_mh,
                    train_sids     = train_sids,
                    train_genders  = train_genders,
                    val_x          = val_x,
                    val_y_mh       = val_y_mh,
                    val_sids       = val_sids,
                    val_genders    = val_genders,
                    attacker_type  = 'adaptive',
                    eval_condition = condition,
                    eval_split     = 'val',
                )
                # Note: adaptive test eval is run automatically
                # inside train_and_evaluate for attacker_type='adaptive'

            # ── B. BLIND — train on GS, eval on raw test ──────────────────────
            if condition == 'raw':
                pass  # no blind for raw
            else:
                if check_completion(arch, pretrained, condition,
                                    attacker_type='blind',
                                    eval_split='test'):
                    print(f"\n⏩ Skip blind test: "
                          f"{arch} {init_str} {condition}")
                else:
                    train_and_evaluate(
                        arch           = arch,
                        pretrained     = pretrained,
                        condition      = condition,
                        train_x        = train_x,
                        train_y_mh     = train_y_mh,
                        train_sids     = train_sids,
                        train_genders  = train_genders,
                        val_x          = TEST_X_RAW,
                        val_y_mh       = TEST_Y_MH_RAW,
                        val_sids       = TEST_SIDS_RAW,
                        val_genders    = TEST_GENDERS_RAW,
                        attacker_type  = 'blind',
                        eval_condition = 'raw',
                        eval_split     = 'test',
                    )

    # ── Release condition tensors ─────────────────────────────────────────────
    del train_x, train_y_int, train_y_mh, val_x, val_y_int, val_y_mh
    torch.cuda.empty_cache()
    gc.collect()
    print(f"\n  ✓ Released {condition} tensors from memory")

print("\n" + "=" * 80)
print("✓ ALL EXPERIMENTS COMPLETE")
print("=" * 80)

summary = get_results_summary()
if summary is not None:
    print("\n📊 RESULTS SUMMARY:")
    print(summary)

=== PRE-FLIGHT CHECKS ===

  ✓ train_raw.pt                        (4.95 GB)
  ✓ train_gs50.pt                       (4.95 GB)
  ✓ train_gs40.pt                       (4.95 GB)
  ✓ train_gs30.pt                       (4.95 GB)
  ✓ train_gs20.pt                       (4.95 GB)
  ✓ train_gs10.pt                       (4.95 GB)
  ✓ train_gs0.pt                        (4.95 GB)
  ✓ val_raw.pt                          (0.75 GB)
  ✓ val_gs50.pt                         (0.75 GB)
  ✓ val_gs40.pt                         (0.75 GB)
  ✓ val_gs30.pt                         (0.75 GB)
  ✓ val_gs20.pt                         (0.75 GB)
  ✓ val_gs10.pt                         (0.75 GB)
  ✓ val_gs0.pt                          (0.75 GB)
  ✓ test_raw.pt                         (1.37 GB)

  test_raw in memory : torch.Size([6802, 1, 224, 224]) ✓

  No completed runs yet — fresh start

Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n2    2.0T  967G 1002G  50% /home/jupyter

✓ Pre-flight complete —

In [ ]:
print('done')

done


## Results

In [ ]:
# =============================================================================
# RESULTS LOADING & PREPARATION
# =============================================================================
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

FIG_DIR = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Load results
df = pd.read_csv(PER_RUN_CSV)

# Clean up / type cast
for col in ['macro_auc', 'male_macro_auc', 'female_macro_auc',
            'bal_acc', 'top1_acc', 'male_bal_acc', 'female_bal_acc']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Condition order — reverse (max drop first) for x-axis
COND_ORDER  = ['raw', 'gs50', 'gs40', 'gs30', 'gs20', 'gs10', 'gs0']
COND_XVALS  = {c: i for i, c in enumerate(COND_ORDER)}
COND_LABELS = ['Raw', 'GS-50', 'GS-40', 'GS-30', 'GS-20', 'GS-10', 'GS-0']

# Per-class AUC column names
CLASS_AUC_COLS = [f'auc_{s}'        for s in SHORT_LIST]
MALE_AUC_COLS  = [f'male_auc_{s}'   for s in SHORT_LIST]
FEM_AUC_COLS   = [f'female_auc_{s}' for s in SHORT_LIST]

# Colour palette
ARCH_COLORS   = {'resnet18': '#4C72B0', 'densenet121': '#DD8452'}
INIT_STYLES   = {'pretrained': '-',     'scratch': '--'}
GENDER_COLORS = {'male': '#5B8DB8',     'female': '#C0616B'}
COND_COLORS   = plt.cm.RdYlGn(
    np.linspace(0.15, 0.85, len(COND_ORDER))
)

print(f"Loaded {len(df)} result rows")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

print(f"\nRun counts:")
print(df.groupby(
    ['arch', 'init', 'attacker_type', 'eval_split']
).size().to_string())

print(f"\nMacro AUC summary:")
print(df.groupby(
    ['arch', 'init', 'train_cond', 'attacker_type', 'eval_split']
)['macro_auc'].first().round(4).to_string())

Loaded 80 result rows

Columns (50):
['arch', 'init', 'train_cond', 'eval_cond', 'attacker_type', 'eval_split', 'best_epoch', 'best_val_auc', 'macro_auc', 'bal_acc', 'top1_acc', 'male_macro_auc', 'female_macro_auc', 'male_n', 'female_n', 'male_bal_acc', 'female_bal_acc', 'auc_NoFind', 'auc_Infilt', 'auc_Atelec', 'auc_Effus', 'auc_Nodule', 'auc_PneuTx', 'auc_Mass', 'auc_Consol', 'auc_PlThck', 'auc_CardMeg', 'auc_Emphy', 'male_auc_NoFind', 'male_auc_Infilt', 'male_auc_Atelec', 'male_auc_Effus', 'male_auc_Nodule', 'male_auc_PneuTx', 'male_auc_Mass', 'male_auc_Consol', 'male_auc_PlThck', 'male_auc_CardMeg', 'male_auc_Emphy', 'female_auc_NoFind', 'female_auc_Infilt', 'female_auc_Atelec', 'female_auc_Effus', 'female_auc_Nodule', 'female_auc_PneuTx', 'female_auc_Mass', 'female_auc_Consol', 'female_auc_PlThck', 'female_auc_CardMeg', 'female_auc_Emphy']

Run counts:
arch         init        attacker_type  eval_split
densenet121  pretrained  adaptive       test          7
                       

In [ ]:
# =============================================================================
# FIGURE 1 — Macro AUC vs GS Condition (Adaptive, val + test)
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharey=False)
fig.suptitle(
    'Model Utility — Macro AUC vs GS Condition (Adaptive Evaluation)\n'
    'Left: Val  |  Right: Test',
    fontsize=14, fontweight='bold'
)

for row_idx, eval_split in enumerate(['val', 'test']):
    adaptive_df = df[
        (df['attacker_type'] == 'adaptive') &
        (df['eval_split']    == eval_split)
    ].copy()
    adaptive_df['x'] = adaptive_df['train_cond'].map(COND_XVALS)

    for col_idx, init in enumerate(['pretrained', 'scratch']):
        ax  = axes[row_idx][col_idx]
        sub = adaptive_df[adaptive_df['init'] == init]

        for arch in ARCHITECTURES:
            arch_sub = sub[sub['arch'] == arch].sort_values('x')
            if arch_sub.empty:
                continue
            ax.plot(
                arch_sub['x'], arch_sub['macro_auc'],
                marker='o', linewidth=2.5, markersize=8,
                color=ARCH_COLORS[arch], label=arch
            )
            for _, r in arch_sub.iterrows():
                ax.annotate(
                    f"{r['macro_auc']:.3f}",
                    (r['x'], r['macro_auc']),
                    textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=7.5
                )

        # Raw baseline
        raw_vals = sub[sub['train_cond'] == 'raw']['macro_auc'].values
        if len(raw_vals):
            ax.axhline(
                raw_vals.mean(), color='gray', linestyle=':',
                linewidth=1.5, label='Raw baseline'
            )

        ax.set_title(
            f'{eval_split.capitalize()} | {init.capitalize()}',
            fontsize=11, fontweight='bold'
        )
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Training Condition')
        ax.set_ylabel('Macro Binary AUC (subject-level)')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

        if not adaptive_df.empty:
            ymin = max(0,   adaptive_df['macro_auc'].min() - 0.05)
            ymax = min(1.0, adaptive_df['macro_auc'].max() + 0.08)
            ax.set_ylim(ymin, ymax)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig1_adaptive_macro_auc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig1_adaptive_macro_auc.png")

Saved → fig1_adaptive_macro_auc.png


In [ ]:
# =============================================================================
# FIGURE 2 — Blind Evaluation: AUC when trained on GS, tested on Raw
# =============================================================================

blind_df      = df[df['attacker_type'] == 'blind'].copy()
blind_df['x'] = blind_df['train_cond'].map(COND_XVALS)

# Raw adaptive baseline from val split
raw_ref = df[
    (df['attacker_type'] == 'adaptive') &
    (df['train_cond']    == 'raw') &
    (df['eval_split']    == 'val')
][['arch', 'init', 'macro_auc']].rename(columns={'macro_auc': 'raw_auc'})

blind_df       = blind_df.merge(raw_ref, on=['arch', 'init'], how='left')
blind_df['auc_drop'] = blind_df['raw_auc'] - blind_df['macro_auc']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    'Blind Evaluation — Trained on GS, Tested on Raw\n'
    'Left: Absolute AUC | Right: AUC Drop vs Raw Val Baseline',
    fontsize=14, fontweight='bold'
)

for row_idx, init in enumerate(['pretrained', 'scratch']):
    ax_abs  = axes[row_idx][0]
    ax_drop = axes[row_idx][1]
    sub     = blind_df[blind_df['init'] == init]

    for arch in ARCHITECTURES:
        arch_sub = sub[sub['arch'] == arch].sort_values('x')
        if arch_sub.empty:
            continue

        ax_abs.plot(
            arch_sub['x'], arch_sub['macro_auc'],
            marker='s', linewidth=2.5, markersize=8,
            color=ARCH_COLORS[arch], label=arch
        )
        ax_drop.bar(
            arch_sub['x'] + (0.2 if arch == 'densenet121' else -0.2),
            arch_sub['auc_drop'],
            width=0.35,
            color=ARCH_COLORS[arch], alpha=0.8,
            label=arch
        )

    # Raw baseline reference lines on absolute plot
    for _, rrow in raw_ref[raw_ref['init'] == init].iterrows():
        ax_abs.axhline(
            rrow['raw_auc'],
            color=ARCH_COLORS[rrow['arch']],
            linestyle=':', linewidth=1.5, alpha=0.6
        )

    ax_drop.axhline(0, color='black', linewidth=0.8)

    for ax, title, ylabel in [
        (ax_abs,  f'{init.capitalize()} — Blind AUC',
         'Macro Binary AUC'),
        (ax_drop, f'{init.capitalize()} — AUC Drop (Raw − Blind)',
         'AUC Drop'),
    ]:
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Training Condition (GS only)')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig2_blind_auc_drop.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig2_blind_auc_drop.png")

Saved → fig2_blind_auc_drop.png


In [ ]:
# =============================================================================
# FIGURE 3 — Gender-Stratified AUC across GS Conditions
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    'Gender-Stratified Macro AUC vs GS Condition\n'
    'Solid=Male  Dashed=Female  Shaded=Gender Gap',
    fontsize=14, fontweight='bold'
)

ATTACKER_SPLIT = {
    'adaptive': 'val',
    'blind':    'test',
}

for row_idx, attacker in enumerate(['adaptive', 'blind']):
    split_filter = ATTACKER_SPLIT[attacker]
    sub_att = df[
        (df['attacker_type'] == attacker) &
        (df['eval_split']    == split_filter)
    ].copy()
    sub_att['x'] = sub_att['train_cond'].map(COND_XVALS)

    for col_idx, init in enumerate(['pretrained', 'scratch']):
        ax  = axes[row_idx][col_idx]
        sub = sub_att[sub_att['init'] == init]

        for arch in ARCHITECTURES:
            arch_sub = sub[sub['arch'] == arch].sort_values('x')
            if arch_sub.empty:
                continue
            xs    = arch_sub['x'].values
            m_auc = arch_sub['male_macro_auc'].values.astype(float)
            f_auc = arch_sub['female_macro_auc'].values.astype(float)

            ax.plot(xs, m_auc, marker='o', linewidth=2,
                    linestyle='-', color=ARCH_COLORS[arch],
                    label=f'{arch} Male')
            ax.plot(xs, f_auc, marker='s', linewidth=2,
                    linestyle='--', color=ARCH_COLORS[arch],
                    label=f'{arch} Female', alpha=0.8)
            ax.fill_between(
                xs, m_auc, f_auc,
                alpha=0.08, color=ARCH_COLORS[arch]
            )

        ax.set_title(
            f'{attacker.capitalize()} ({split_filter}) | '
            f'{init.capitalize()}',
            fontsize=11, fontweight='bold'
        )
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Condition')
        ax.set_ylabel('Macro Binary AUC')
        ax.legend(fontsize=8, ncol=2)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig3_gender_auc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig3_gender_auc.png")

Saved → fig3_gender_auc.png


In [ ]:
# =============================================================================
# FIGURE 4 — Gender Gap Heatmap (|Male AUC − Female AUC|)
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
fig.suptitle(
    'Gender Gap Magnitude  |Male AUC − Female AUC|\n'
    'Larger = More Disparate Performance',
    fontsize=14, fontweight='bold'
)

row_keys = [
    f'{arch}_{init}'
    for init in ['pretrained', 'scratch']
    for arch in ARCHITECTURES
]

for ax, attacker in zip(axes, ['adaptive', 'blind']):
    split_filter = ATTACKER_SPLIT[attacker]
    sub = df[
        (df['attacker_type'] == attacker) &
        (df['eval_split']    == split_filter)
    ].copy()

    gap_matrix = np.full((len(row_keys), len(COND_ORDER)), np.nan)

    for r_idx, row_key in enumerate(row_keys):
        arch, init = row_key.rsplit('_', 1)
        for c_idx, cond in enumerate(COND_ORDER):
            match = sub[
                (sub['arch']       == arch) &
                (sub['init']       == init) &
                (sub['train_cond'] == cond)
            ]
            if len(match):
                m = float(match['male_macro_auc'].values[0])
                f = float(match['female_macro_auc'].values[0])
                if not (np.isnan(m) or np.isnan(f)):
                    gap_matrix[r_idx, c_idx] = abs(m - f)

    sns.heatmap(
        gap_matrix,
        annot=True, fmt='.3f',
        cmap='YlOrRd',
        xticklabels=COND_LABELS,
        yticklabels=row_keys,
        ax=ax,
        vmin=0, vmax=0.1,
        linewidths=0.5,
        cbar_kws={'label': '|M−F| AUC'}
    )
    ax.set_title(
        f'{attacker.capitalize()} ({split_filter})',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Condition')
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig4_gender_gap_heatmap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig4_gender_gap_heatmap.png")

Saved → fig4_gender_gap_heatmap.png


In [ ]:
# =============================================================================
# FIGURE 5 — Per-Class AUC Heatmap (Adaptive, Val + Test)
# =============================================================================

combos = [
    ('resnet18',    'pretrained'),
    ('resnet18',    'scratch'),
    ('densenet121', 'pretrained'),
    ('densenet121', 'scratch'),
]

for eval_split in ['val', 'test']:
    adaptive_split_df = df[
        (df['attacker_type'] == 'adaptive') &
        (df['eval_split']    == eval_split)
    ].copy()

    if adaptive_split_df.empty:
        print(f"  No adaptive {eval_split} results yet — skipping fig5 {eval_split}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    fig.suptitle(
        f'Per-Class Binary AUC across GS Conditions — '
        f'Adaptive Evaluation ({eval_split.capitalize()})',
        fontsize=14, fontweight='bold'
    )

    for ax, (arch, init) in zip(axes.flat, combos):
        sub = adaptive_split_df[
            (adaptive_split_df['arch'] == arch) &
            (adaptive_split_df['init'] == init)
        ].copy()

        matrix = np.full((len(SHORT_LIST), len(COND_ORDER)), np.nan)

        for c_idx, cond in enumerate(COND_ORDER):
            row = sub[sub['train_cond'] == cond]
            if row.empty:
                continue
            row = row.iloc[0]
            for r_idx, short in enumerate(SHORT_LIST):
                col = f'auc_{short}'
                if col in sub.columns:
                    val = row.get(col, np.nan)
                    matrix[r_idx, c_idx] = float(val) \
                        if not pd.isna(val) else np.nan

        sns.heatmap(
            matrix,
            annot=True, fmt='.3f',
            cmap='RdYlGn',
            xticklabels=COND_LABELS,
            yticklabels=SHORT_LIST,
            ax=ax,
            vmin=0.4, vmax=1.0,
            linewidths=0.4,
            cbar_kws={'label': 'Binary AUC'}
        )
        ax.set_title(f'{arch} — {init}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Condition')
        ax.tick_params(axis='y', labelsize=9)

    plt.tight_layout()
    fname = f'fig5_perclass_auc_heatmap_{eval_split}.png'
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fname}")

Saved → fig5_perclass_auc_heatmap_val.png
Saved → fig5_perclass_auc_heatmap_test.png


In [ ]:
# =============================================================================
# FIGURE 6 — Per-Class AUC Degradation vs GS Intensity
# =============================================================================

class_colors = plt.cm.tab20.colors[:len(SHORT_LIST)]

for eval_split in ['val', 'test']:
    adaptive_split_df = df[
        (df['attacker_type'] == 'adaptive') &
        (df['eval_split']    == eval_split)
    ].copy()

    if adaptive_split_df.empty:
        print(f"  No adaptive {eval_split} results yet — skipping fig6 {eval_split}")
        continue

    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    fig.suptitle(
        f'Per-Class Binary AUC vs GS Intensity — '
        f'Adaptive ({eval_split.capitalize()})\n'
        'Which pathologies are most affected by GS?',
        fontsize=14, fontweight='bold'
    )

    for ax, (arch, init) in zip(axes.flat, combos):
        sub = adaptive_split_df[
            (adaptive_split_df['arch'] == arch) &
            (adaptive_split_df['init'] == init)
        ].copy()
        sub['x'] = sub['train_cond'].map(COND_XVALS)
        sub = sub.sort_values('x')

        for r_idx, short in enumerate(SHORT_LIST):
            col = f'auc_{short}'
            if col not in sub.columns:
                continue
            vals = pd.to_numeric(sub[col], errors='coerce').values
            xs   = sub['x'].values
            ax.plot(
                xs, vals,
                marker='o', linewidth=1.8, markersize=6,
                color=class_colors[r_idx],
                label=short, alpha=0.85
            )

        ax.set_title(f'{arch} — {init}',
                     fontsize=11, fontweight='bold')
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Condition')
        ax.set_ylabel('Binary AUC')
        ax.legend(fontsize=7.5, ncol=3, loc='lower left')
        ax.grid(alpha=0.3)
        ax.set_ylim(0.3, 1.05)

    plt.tight_layout()
    fname = f'fig6_perclass_degradation_{eval_split}.png'
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved → {fname}")

Saved → fig6_perclass_degradation_val.png
Saved → fig6_perclass_degradation_test.png


In [ ]:
# =============================================================================
# FIGURE 7 — Per-Class Gender Gap Heatmap
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(22, 8))
fig.suptitle(
    'Per-Class Gender Gap  |Male AUC − Female AUC| per Condition\n'
    'Pretrained Models — Adaptive Evaluation (Val)\n'
    'Darker = Larger Gender Disparity',
    fontsize=13, fontweight='bold'
)

adaptive_val_df = df[
    (df['attacker_type'] == 'adaptive') &
    (df['eval_split']    == 'val')
].copy()

for ax, arch in zip(axes, ARCHITECTURES):
    sub = adaptive_val_df[
        (adaptive_val_df['arch'] == arch) &
        (adaptive_val_df['init'] == 'pretrained')
    ].copy()

    matrix = np.full((len(SHORT_LIST), len(COND_ORDER)), np.nan)

    for c_idx, cond in enumerate(COND_ORDER):
        row = sub[sub['train_cond'] == cond]
        if row.empty:
            continue
        row = row.iloc[0]
        for r_idx, short in enumerate(SHORT_LIST):
            m_col = f'male_auc_{short}'
            f_col = f'female_auc_{short}'
            if m_col in sub.columns and f_col in sub.columns:
                m_v = row.get(m_col, np.nan)
                f_v = row.get(f_col, np.nan)
                if not (pd.isna(m_v) or pd.isna(f_v)):
                    matrix[r_idx, c_idx] = abs(float(m_v) - float(f_v))

    sns.heatmap(
        matrix,
        annot=True, fmt='.3f',
        cmap='YlOrRd',
        xticklabels=COND_LABELS,
        yticklabels=SHORT_LIST,
        ax=ax,
        vmin=0, vmax=0.15,
        linewidths=0.4,
        cbar_kws={'label': '|M−F| AUC'}
    )
    ax.set_title(f'{arch} — pretrained',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Condition')
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_perclass_gender_gap.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig7_perclass_gender_gap.png")

Saved → fig7_perclass_gender_gap.png


In [ ]:
# =============================================================================
# FIGURE 8 — Balanced Accuracy & Top-1 Accuracy vs Condition
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle(
    'Balanced Accuracy & Top-1 Accuracy vs GS Condition\n'
    'Adaptive Evaluation (Val)',
    fontsize=14, fontweight='bold'
)

metrics_to_plot = [
    ('bal_acc',  'Balanced Accuracy'),
    ('top1_acc', 'Top-1 Accuracy'),
]

adaptive_val_df = df[
    (df['attacker_type'] == 'adaptive') &
    (df['eval_split']    == 'val')
].copy()

for col_idx, (metric_col, metric_label) in enumerate(metrics_to_plot):
    for row_idx, init in enumerate(['pretrained', 'scratch']):
        ax  = axes[row_idx][col_idx]
        sub = adaptive_val_df[adaptive_val_df['init'] == init].copy()
        sub['x'] = sub['train_cond'].map(COND_XVALS)

        for arch in ARCHITECTURES:
            arch_sub = sub[sub['arch'] == arch].sort_values('x')
            if arch_sub.empty:
                continue
            ax.plot(
                arch_sub['x'], arch_sub[metric_col],
                marker='o', linewidth=2.5, markersize=8,
                color=ARCH_COLORS[arch], label=arch
            )
            for _, row in arch_sub.iterrows():
                ax.annotate(
                    f"{row[metric_col]:.3f}",
                    (row['x'], row[metric_col]),
                    textcoords='offset points',
                    xytext=(0, 8), ha='center', fontsize=7
                )

        ax.set_title(
            f'{metric_label} — {init.capitalize()}',
            fontsize=11, fontweight='bold'
        )
        ax.set_xticks(range(len(COND_ORDER)))
        ax.set_xticklabels(COND_LABELS, fontsize=9)
        ax.set_xlabel('Condition')
        ax.set_ylabel(metric_label)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig8_bal_acc_top1.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig8_bal_acc_top1.png")

Saved → fig8_bal_acc_top1.png


In [ ]:
# =============================================================================
# FIGURE 9 — Pretrained vs Scratch AUC Gap
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    'Pretrained vs Scratch — Macro AUC Gap across GS Conditions\n'
    'Positive = Pretrained better | Negative = Scratch better',
    fontsize=14, fontweight='bold'
)

for ax, attacker in zip(axes, ['adaptive', 'blind']):
    split_filter = ATTACKER_SPLIT[attacker]
    sub = df[
        (df['attacker_type'] == attacker) &
        (df['eval_split']    == split_filter)
    ].copy()
    sub['x'] = sub['train_cond'].map(COND_XVALS)

    for arch in ARCHITECTURES:
        arch_sub = sub[sub['arch'] == arch]
        pre = arch_sub[
            arch_sub['init'] == 'pretrained'
        ].sort_values('x')
        scr = arch_sub[
            arch_sub['init'] == 'scratch'
        ].sort_values('x')

        merged = pre[['x', 'train_cond', 'macro_auc']].merge(
            scr[['x', 'train_cond', 'macro_auc']],
            on=['x', 'train_cond'],
            suffixes=('_pre', '_scr')
        )
        if merged.empty:
            continue
        merged['gap'] = (
            merged['macro_auc_pre'] - merged['macro_auc_scr']
        )

        ax.plot(
            merged['x'], merged['gap'],
            marker='o', linewidth=2.5, markersize=8,
            color=ARCH_COLORS[arch], label=arch
        )
        ax.fill_between(
            merged['x'], merged['gap'], 0,
            alpha=0.08, color=ARCH_COLORS[arch]
        )

    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_title(
        f'{attacker.capitalize()} ({split_filter})',
        fontsize=12, fontweight='bold'
    )
    ax.set_xticks(range(len(COND_ORDER)))
    ax.set_xticklabels(COND_LABELS, fontsize=9)
    ax.set_xlabel('Condition')
    ax.set_ylabel('AUC Gap (Pretrained − Scratch)')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig9_pretrained_vs_scratch.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig9_pretrained_vs_scratch.png")

Saved → fig9_pretrained_vs_scratch.png


In [ ]:
# =============================================================================
# FIGURE 10 — Summary Dashboard
# =============================================================================

fig = plt.figure(figsize=(24, 20))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.5, wspace=0.4)
fig.suptitle(
    'NIH ChestX-ray14 — GS Utility Experiment Summary Dashboard\n'
    f'ResNet18 & DenseNet121 | 11 Classes | {len(df)} total runs',
    fontsize=15, fontweight='bold', y=1.01
)

adap_val = df[
    (df['attacker_type'] == 'adaptive') &
    (df['eval_split']    == 'val')
].copy()
adap_val['x'] = adap_val['train_cond'].map(COND_XVALS)

# ── 1. Adaptive AUC pretrained ────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
sub = adap_val[adap_val['init'] == 'pretrained']
for arch in ARCHITECTURES:
    a = sub[sub['arch'] == arch].sort_values('x')
    ax1.plot(a['x'], a['macro_auc'], marker='o',
             color=ARCH_COLORS[arch], label=arch, linewidth=2)
ax1.set_title('Adaptive AUC (Pretrained, Val)',
              fontweight='bold', fontsize=10)
ax1.set_xticks(range(len(COND_ORDER)))
ax1.set_xticklabels(COND_LABELS, fontsize=7, rotation=30)
ax1.set_ylabel('Macro Binary AUC')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# ── 2. Adaptive AUC scratch ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
sub = adap_val[adap_val['init'] == 'scratch']
for arch in ARCHITECTURES:
    a = sub[sub['arch'] == arch].sort_values('x')
    ax2.plot(a['x'], a['macro_auc'], marker='o', linestyle='--',
             color=ARCH_COLORS[arch], label=arch, linewidth=2)
ax2.set_title('Adaptive AUC (Scratch, Val)',
              fontweight='bold', fontsize=10)
ax2.set_xticks(range(len(COND_ORDER)))
ax2.set_xticklabels(COND_LABELS, fontsize=7, rotation=30)
ax2.set_ylabel('Macro Binary AUC')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# ── 3. Blind AUC pretrained ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
blind_sub = df[
    (df['attacker_type'] == 'blind') &
    (df['eval_split']    == 'test') &
    (df['init']          == 'pretrained')
].copy()
blind_sub['x'] = blind_sub['train_cond'].map(COND_XVALS)
for arch in ARCHITECTURES:
    a = blind_sub[blind_sub['arch'] == arch].sort_values('x')
    if a.empty:
        continue
    ax3.plot(a['x'], a['macro_auc'], marker='s',
             color=ARCH_COLORS[arch], label=arch, linewidth=2)
ax3.set_title('Blind AUC (Pretrained, Test)',
              fontweight='bold', fontsize=10)
ax3.set_xticks(range(len(COND_ORDER)))
ax3.set_xticklabels(COND_LABELS, fontsize=7, rotation=30)
ax3.set_ylabel('Macro Binary AUC')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

# ── 4. Gender gap pretrained adaptive val ─────────────────────────────────────
ax4 = fig.add_subplot(gs[1, :2])
sub = adap_val[adap_val['init'] == 'pretrained']
for arch in ARCHITECTURES:
    a = sub[sub['arch'] == arch].sort_values('x')
    if a.empty:
        continue
    ax4.plot(a['x'], a['male_macro_auc'].astype(float),
             marker='o', linewidth=2,
             color=ARCH_COLORS[arch], linestyle='-',
             label=f'{arch} M')
    ax4.plot(a['x'], a['female_macro_auc'].astype(float),
             marker='s', linewidth=2,
             color=ARCH_COLORS[arch], linestyle='--',
             label=f'{arch} F', alpha=0.8)
    ax4.fill_between(
        a['x'],
        a['male_macro_auc'].astype(float),
        a['female_macro_auc'].astype(float),
        alpha=0.07, color=ARCH_COLORS[arch]
    )
ax4.set_title('Gender AUC Gap — Pretrained Adaptive (Val)',
              fontweight='bold', fontsize=10)
ax4.set_xticks(range(len(COND_ORDER)))
ax4.set_xticklabels(COND_LABELS, fontsize=8, rotation=30)
ax4.set_ylabel('Macro Binary AUC')
ax4.legend(fontsize=8, ncol=4)
ax4.grid(alpha=0.3)

# ── 5. Pretrained vs scratch gap ──────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
for arch in ARCHITECTURES:
    pre = adap_val[
        (adap_val['arch'] == arch) &
        (adap_val['init'] == 'pretrained')
    ].sort_values('x')
    scr = adap_val[
        (adap_val['arch'] == arch) &
        (adap_val['init'] == 'scratch')
    ].sort_values('x')
    m = pre[['x', 'macro_auc']].merge(
        scr[['x', 'macro_auc']], on='x', suffixes=('_p', '_s')
    )
    if m.empty:
        continue
    ax5.plot(m['x'], m['macro_auc_p'] - m['macro_auc_s'],
             marker='o', linewidth=2,
             color=ARCH_COLORS[arch], label=arch)
ax5.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax5.set_title('Pretrained − Scratch Gap (Val)',
              fontweight='bold', fontsize=10)
ax5.set_xticks(range(len(COND_ORDER)))
ax5.set_xticklabels(COND_LABELS, fontsize=7, rotation=30)
ax5.set_ylabel('AUC Gap')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.3)

# ── 6. Per-class AUC bar — raw pretrained val ─────────────────────────────────
ax6 = fig.add_subplot(gs[2, :])
raw_pre = adap_val[
    (adap_val['train_cond'] == 'raw') &
    (adap_val['init']       == 'pretrained')
]
x6     = np.arange(len(SHORT_LIST))
width6 = 0.35
for i, arch in enumerate(ARCHITECTURES):
    row = raw_pre[raw_pre['arch'] == arch]
    if row.empty:
        continue
    row  = row.iloc[0]
    vals = [pd.to_numeric(row.get(f'auc_{s}', np.nan),
                          errors='coerce')
            for s in SHORT_LIST]
    ax6.bar(
        x6 + (i - 0.5) * width6, vals, width6,
        label=arch, color=ARCH_COLORS[arch], alpha=0.85
    )
ax6.set_xticks(x6)
ax6.set_xticklabels(SHORT_LIST, fontsize=9)
ax6.set_ylabel('Binary AUC')
ax6.set_title('Per-Class AUC — Raw Condition, Pretrained (Val)',
              fontweight='bold', fontsize=10)
ax6.axhline(0.5, color='red', linestyle=':', linewidth=1,
            label='Chance (0.5)')
ax6.legend(fontsize=9)
ax6.grid(axis='y', alpha=0.3)
ax6.set_ylim(0.3, 1.05)

plt.savefig(FIG_DIR / 'fig10_summary_dashboard.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Saved → fig10_summary_dashboard.png")

print(f"\n=== ALL FIGURES SAVED TO {FIG_DIR} ===")
for f in sorted(FIG_DIR.iterdir()):
    print(f"  {f.name}")

Saved → fig10_summary_dashboard.png

=== ALL FIGURES SAVED TO results/nih_cxr/utility/figures ===
  fig10_summary_dashboard.png
  fig1_adaptive_macro_auc.png
  fig2_blind_auc_drop.png
  fig3_gender_auc.png
  fig4_gender_gap_heatmap.png
  fig5_perclass_auc_heatmap_test.png
  fig5_perclass_auc_heatmap_val.png
  fig6_perclass_degradation_test.png
  fig6_perclass_degradation_val.png
  fig7_perclass_gender_gap.png
  fig8_bal_acc_top1.png
  fig9_pretrained_vs_scratch.png
